# Libs

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, balanced_accuracy_score, roc_curve, roc_auc_score
from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
import plotly.graph_objects as go
from utils.futurai_ppd import drop_transitorio_desligado
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelBinarizer
from sklearn.linear_model import RidgeClassifierCV
from sktime.transformations.panel.rocket import Rocket
from collections import Counter
from tsfresh import extract_features
from tsfresh.utilities.dataframe_functions import impute
from tsfresh import select_features
from scipy.stats import kurtosis
from sktime.datatypes._panel._convert import from_long_to_nested

import warnings
warnings.filterwarnings('ignore')

# Import dataset

In [ ]:
base_name = 'Depurador 762-28-006 - Cozimento'
timestamp = "Timestamp"

df_dataset = pd.read_csv('data/' + base_name + '.csv', sep=";", decimal=".", encoding="utf-8-sig")
df_dataset[timestamp] = pd.to_datetime(df_dataset[timestamp], format="%Y-%m-%d %H:%M:%S")

## Drop columns with NaN values, constant values or irrelevant to the analysis
df_dataset.drop(columns=["762H0336.PV", "762H0342.PV", "762N0015.SP", "762P0013.SP", "762-34-073.CR", "762N0015.OP", "762F0014.SP"], inplace=True, errors='ignore')

df_dataset.dropna(inplace=True)

print(f"Dataset shape: {df_dataset.shape}")

list_variables = df_dataset.columns.tolist()
df_dataset.head()

## Remove periods Off

In [ ]:
pre_process = []
pp_var_ref_desligado = "762-28-006.CR"
pp_valor_ref_desligado = 5
pp_tempo_ref_desligado = 0
pp_pre_corte_transitorio = 30
pp_pos_corte_transitorio = 30
pre_process.append(  
{
   "after_cut": pp_pos_corte_transitorio,
   "interval_off": pp_tempo_ref_desligado,
   "limit_off": pp_valor_ref_desligado,
   "pre_cut": pp_pre_corte_transitorio,
   "variable_off": pp_var_ref_desligado
  })

for pro in pre_process:
    df_dataset,_,_ = drop_transitorio_desligado(df_dataset,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
print(f"Dataset shape: {df_dataset.shape}")
df_dataset.head()

## Create label for anomaly

In [ ]:
periodos_de_Falhas = [
    (pd.Timestamp('2024-05-03 11:00:00'), pd.Timestamp('2024-05-03 11:35:00')),
    (pd.Timestamp('2024-06-25 17:20:00'), pd.Timestamp('2024-08-02 14:00:00')),
    (pd.Timestamp('2024-10-19 10:40:00'), pd.Timestamp('2024-10-19 10:50:00')),
    (pd.Timestamp('2024-10-19 10:40:00'), pd.Timestamp('2024-10-19 10:50:00')),
    (pd.Timestamp('2024-10-21 12:00:00'), pd.Timestamp('2024-10-22 00:35:00')),
    (pd.Timestamp('2024-10-24 03:10:00'), pd.Timestamp('2024-10-27 00:00:00')),
    (pd.Timestamp('2024-11-14 06:40:00'), pd.Timestamp('2024-11-14 19:45:00')),
    (pd.Timestamp('2024-11-25 21:45:00'), pd.Timestamp('2024-11-25 22:03:00')),
    (pd.Timestamp('2024-11-27 15:00:00'), pd.Timestamp('2024-11-27 15:07:00')),
    (pd.Timestamp('2024-11-27 16:04:00'), pd.Timestamp('2024-11-27 16:11:00')),
    (pd.Timestamp('2024-11-30 13:30:00'), pd.Timestamp('2024-11-30 15:52:00')),
    (pd.Timestamp('2024-12-09 20:30:00'), pd.Timestamp('2024-12-11 07:00:00')),
    (pd.Timestamp('2024-12-12 20:45:00'), pd.Timestamp('2024-12-12 21:15:00')),
    (pd.Timestamp('2025-03-05 15:17:00'), pd.Timestamp('2025-03-05 15:24:00')),
    (pd.Timestamp('2025-03-15 18:30:00'), pd.Timestamp('2025-03-15 19:15:00')),
    (pd.Timestamp('2025-03-18 11:40:00'), pd.Timestamp('2025-03-18 20:00:00')),
    (pd.Timestamp('2025-06-11 17:25:00'), pd.Timestamp('2025-06-11 17:50:00')),
    (pd.Timestamp('2025-06-16 15:20:00'), pd.Timestamp('2025-06-17 07:38:00')),
]

df_dataset['Falhas'] = 0
for inicio, fim in periodos_de_Falhas:
    df_dataset.loc[(df_dataset[timestamp] >= inicio) & (df_dataset[timestamp] <= fim), 'Falhas'] = 1
df_dataset.head()

## Import TAGs and descriptions

In [ ]:
df_subsistema = pd.read_csv('data/'+ base_name + '_subsistema.csv', sep=";", decimal=".", encoding="utf-8-sig")
df_subsistema

## Plot variables

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_dataset['Timestamp'],
    y=df_dataset["762P0034.PV"],
    mode='lines',
    name='762P0034.PV',
    line=dict(color='black')
))

fig.add_trace(go.Scatter(
    x=df_dataset['Timestamp'],
    y=df_dataset["Falhas"],
    mode='lines',
    name='Falhas',
    line=dict(color='red')
))

fig.update_layout(
    template='plotly_white',
    hovermode='x unified'
)
fig.show()

# TSC with SKTIME Algorithms

## ROCKET

### Split train/test data

#### Train

In [ ]:
start_date = pd.to_datetime("2024-01-01 00:00:00")
end_date = pd.to_datetime("2025-03-12 09:25:00")
mask = (df_dataset[timestamp] >= start_date) & (df_dataset[timestamp] <= end_date)
df_train = df_dataset.loc[mask]

df_train.sort_values(by=[timestamp],inplace=True)
print(df_train.shape)
print("Falhas Train:",df_train[df_train["Falhas"]==1].shape[0])
print("Normal Train:",df_train[df_train["Falhas"]==0].shape[0])

#### Test

In [ ]:
start_date = pd.to_datetime("2025-03-14 07:54:00")
end_date = pd.to_datetime("2025-09-19 15:00:00")
mask = (df_dataset[timestamp] >= start_date) & (df_dataset[timestamp] <= end_date)
df_test = df_dataset.loc[mask]

df_test.sort_values(by=[timestamp],inplace=True)
print(df_test.shape)
print("Falhas Train:",df_test[df_test["Falhas"]==1].shape[0])
print("Normal Train:",df_test[df_test["Falhas"]==0].shape[0])

### Data Scaling

In [ ]:
X_train = df_train.drop(columns=["Timestamp","Falhas"], axis=1)
y_train = df_train["Falhas"]

X_test = df_test.drop(columns=["Timestamp","Falhas"], axis=1)
y_test = df_test["Falhas"]

scaler = StandardScaler()
scaler.fit(X_train)

X_train = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

### Format data to sktime

In [ ]:
def format_df_for_sktime(
    df: pd.DataFrame,
    timestamp_col: str,
    label_col: str,
    window_duration_minutes: int,
    overlap_duration_minutes: int
) -> (pd.DataFrame, np.ndarray):
    """
    Formata um DataFrame de série temporal para o formato aninhado da sktime
    usando janelamento com sobreposição.

    Args:
        df: DataFrame original.
        timestamp_col: Nome da coluna de timestamp.
        label_col: Nome da coluna de rótulo.
        window_duration_minutes: Duração da janela em minutos.
        overlap_duration_minutes: Duração da sobreposição em minutos.

    Returns:
        Uma tupla contendo:
        - X_nested: DataFrame formatado para sktime.
        - y: Array numpy com os rótulos de cada janela.
    """
    # Isola as colunas de features
    feature_cols = [col for col in df.columns if col != label_col]
    features_df = df[feature_cols]
    labels_series = df[label_col]

    # Converte durações para o formato Timedelta do pandas
    window_delta = pd.Timedelta(minutes=window_duration_minutes)
    step_delta = pd.Timedelta(minutes=(window_duration_minutes - overlap_duration_minutes))

    window_list = []
    label_list = []

    start_time = df.index.min()
    end_time = df.index.max()
    
    current_time = start_time
    
    while current_time + window_delta <= end_time:
        window_end = current_time + window_delta
        
        # Seleciona os dados da janela
        window_features = features_df.loc[current_time:window_end]
        window_labels = labels_series.loc[current_time:window_end]
        
        # Garante que a janela tenha o tamanho esperado (ignora janelas no final que possam ser menores)
        # Uma janela de 60 minutos com freq 'T' deve ter 61 pontos (incluindo o início e o fim)
        if len(window_features) == window_duration_minutes + 1:
            window_list.append(window_features)
            
            # Lógica de rotulagem: 1 se a maioria das amostras na janela for 1
            label = 1 if (window_labels.mean() > 0.5) else 0
            label_list.append(label)

        current_time += step_delta
    
    if not window_list:
        print("Nenhuma janela foi criada. Verifique a duração dos dados e os parâmetros da janela.")
        return pd.DataFrame(), np.array([])

    # Monta o DataFrame aninhado para sktime
    sktime_data = {}
    for col in feature_cols:
        # Para cada feature, cria uma lista de Series, onde cada Series corresponde a uma janela
        sktime_data[col] = [win[col] for win in window_list]

    X_nested = pd.DataFrame(sktime_data)
    y = np.array(label_list)
    
    return X_nested, y

#### Train

In [ ]:
WINDOW_MINUTES = 60
OVERLAP_MINUTES = 30

df_train = df_train.set_index(timestamp)
X_train_sktime, y_train_sktime = format_df_for_sktime(
    df=df_train,
    timestamp_col=timestamp,
    label_col='Falhas',
    window_duration_minutes=WINDOW_MINUTES,
    overlap_duration_minutes=OVERLAP_MINUTES
)

print("X_train_sktime shape:", X_train_sktime.shape)
print("y_train_sktime shape:", y_train_sktime.shape)

print("\nLabels distribution in train:")
print(pd.Series(y_train_sktime).value_counts())

#### Test

In [ ]:
df_test = df_test.set_index(timestamp)

WINDOW_MINUTES = 60
OVERLAP_MINUTES = 30

X_test_sktime, y_test_sktime = format_df_for_sktime(
    df=df_test,
    timestamp_col=timestamp,
    label_col='Falhas',
    window_duration_minutes=WINDOW_MINUTES,
    overlap_duration_minutes=OVERLAP_MINUTES
)

print("X_test_sktime shape:", X_test_sktime.shape)
print("y_test_sktime shape:", y_test_sktime.shape)

print("\nLabels distribution in test:")
print(pd.Series(y_test_sktime).value_counts())

### ROCKET

In [ ]:
rocket = Rocket(num_kernels=10000, normalise=True, random_state=42)
rocket.fit(X_train_sktime)

X_train_rocket = rocket.transform(X_train_sktime)
X_test_rocket = rocket.transform(X_test_sktime)

print("Shape features após Rocket:", X_train_rocket.shape)

#### Fit model with RidgeClassifierCV

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(y_train_sktime)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_sktime
)
class_weights = dict(zip(classes, class_weights))

## RidgeClassifierCV is recommended by the ROCKET paper
rdgcv_clf = RidgeClassifierCV(alphas=np.logspace(-5, 5, 100), class_weight=class_weights)
rdgcv_clf.fit(X_train_rocket, y_train_sktime)

## Predict
y_pred_rocket = rdgcv_clf.predict(X_test_rocket)

accuracy = balanced_accuracy_score(y_test_sktime, y_pred_rocket)
f1 = f1_score(y_test_sktime, y_pred_rocket, average='weighted')
recall = recall_score(y_test_sktime, y_pred_rocket, average='weighted')
precision = precision_score(y_test_sktime, y_pred_rocket, average='weighted')

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Precision: {precision:.4f}")

## Plot ROC Curve
y_scores = rdgcv_clf.decision_function(X_test_rocket)

lb = LabelBinarizer()
y_test_bin = lb.fit_transform(y_test_sktime)

if y_test_bin.shape[1] == 1:
    y_test_bin = y_test_bin.ravel()

fpr, tpr, thresholds = roc_curve(y_test_bin, y_scores)
roc_auc = roc_auc_score(y_test_bin, y_scores)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, color="blue", lw=2, label=f"ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

#### Plot predictions X real labels

In [ ]:
# y_test_sktime.sort_index(inplace=True)

fig_pred = go.Figure()
fig_pred.add_trace(go.Scatter(
    y=y_pred_rocket,
    name='Predict',
    mode="lines",
    line=dict(color='black')
))
fig_pred.add_trace(go.Scatter(
    y=y_test_sktime,
    name='Real',
    mode="lines",
    line=dict(color='red')
))
fig_pred.show()

### MiniROCKET

In [ ]:
from sktime.transformations.panel.rocket import MiniRocketMultivariate

minirocket = MiniRocketMultivariate(num_kernels=10000, max_dilations_per_kernel=32, random_state=42)
minirocket.fit(X_train)

X_train_minirocket = minirocket.transform(X_train_sktime)
X_test_minirocket = minirocket.transform(X_test_sktime)

print("Shape features após Rocket:", X_train_minirocket.shape)

#### Fit model with RidgeClassifierCV

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(y_train_sktime)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_sktime
)
class_weights = dict(zip(classes, class_weights))

## RidgeClassifierCV is recommended by the ROCKET paper
rdgcv_clf = RidgeClassifierCV(alphas=np.logspace(-5, 5, 100), class_weight=class_weights)
rdgcv_clf.fit(X_train_minirocket, y_train_sktime)

## Predict
y_pred_minirocket = rdgcv_clf.predict(X_test_minirocket)

accuracy = balanced_accuracy_score(y_test_sktime, y_pred_minirocket)
f1 = f1_score(y_test_sktime, y_pred_minirocket, average='weighted')
recall = recall_score(y_test_sktime, y_pred_minirocket, average='weighted')
precision = precision_score(y_test_sktime, y_pred_minirocket, average='weighted')

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Precision: {precision:.4f}")

## Plot ROC Curve
y_scores = rdgcv_clf.decision_function(X_test_minirocket)

lb = LabelBinarizer()
y_test_bin = lb.fit_transform(y_test_sktime)

if y_test_bin.shape[1] == 1:
    y_test_bin = y_test_bin.ravel()

fpr, tpr, thresholds = roc_curve(y_test_bin, y_scores)
roc_auc = roc_auc_score(y_test_bin, y_scores)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, color="blue", lw=2, label=f"ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

#### Plot predictions X Real label

In [ ]:
# y_test_sktime.sort_index(inplace=True)

fig_pred = go.Figure()
fig_pred.add_trace(go.Scatter(
    y=y_pred_minirocket,
    name='Predict',
    mode="lines",
    line=dict(color='black')
))
fig_pred.add_trace(go.Scatter(
    y=y_test_sktime,
    name='Real',
    mode="lines",
    line=dict(color='red')
))
fig_pred.show()